# Retention Ranking Analysis

Checkpoint 43 evaluates how effectively the selected sigmoid-calibrated Logistic Regression prioritizes validation-period attrition cases. The 2025 test target remains locked.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
PROCESSED = PROJECT_ROOT / 'data' / 'processed'

ranking = pd.read_csv(PROCESSED / 'retention_ranking_metrics.csv')
summary = pd.read_csv(PROCESSED / 'retention_ranking_summary.csv')
curves = pd.read_csv(PROCESSED / 'retention_curve_points.csv')
confusion = pd.read_csv(PROCESSED / 'retention_top_decile_confusion_matrix.csv')
validation = pd.read_csv(PROCESSED / 'retention_ranking_validation.csv')

print(f'Ranking grid rows: {len(ranking):,}')
print(f"Checks passed: {(validation['status'] == 'PASS').sum()}/{len(validation)}")
summary.round(4)

## Top-k metrics

Precision measures concentration inside the selected group. Capture measures the share of all attrition cases found. Lift compares the selected group with random selection.

In [ ]:
ranking[[
    'requested_fraction',
    'selected_count',
    'captured_positive_cases',
    'precision_at_k',
    'capture_rate_at_k',
    'lift_at_k',
    'precision_at_k_lower_95',
    'precision_at_k_upper_95',
]].round(4)

## Precision-recall curve

In [ ]:
pr = curves.loc[curves['curve_type'].eq('precision_recall')]
baseline = float(summary.loc[0, 'positive_rate'])
figure, axis = plt.subplots(figsize=(8, 5))
axis.plot(pr['x'], pr['y'], linewidth=2, label='Selected model')
axis.axhline(baseline, linestyle='--', color='gray', label=f'No-skill baseline ({baseline:.1%})')
axis.set(xlabel='Recall', ylabel='Precision', title='Validation Precision–Recall Curve', xlim=(0, 1))
axis.legend()
axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Cumulative gains

In [ ]:
gains = curves.loc[curves['curve_type'].eq('cumulative_gains')]
figure, axis = plt.subplots(figsize=(8, 5))
axis.plot(gains['x'], gains['y'], linewidth=2, label='Selected model')
axis.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random selection')
axis.set(
    xlabel='Fraction of workforce reviewed',
    ylabel='Fraction of attrition cases captured',
    title='Validation Cumulative Gains Curve',
    xlim=(0, 1),
    ylim=(0, 1),
)
axis.legend()
axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Lift curve

In [ ]:
lift = curves.loc[curves['curve_type'].eq('lift')]
figure, axis = plt.subplots(figsize=(8, 5))
axis.plot(lift['x'], lift['y'], linewidth=2, label='Selected model')
axis.axhline(1, linestyle='--', color='gray', label='Random-selection lift')
axis.set(
    xlabel='Fraction of workforce reviewed',
    ylabel='Lift over random selection',
    title='Validation Lift Curve',
    xlim=(0, 1),
)
axis.legend()
axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Top-decile reporting matrix

This matrix describes the highest-ranked 10%. It does not choose the final business threshold.

In [ ]:
confusion.pivot(
    index='actual_outcome',
    columns='selection_outcome',
    values='count',
)

The model provides moderate prioritization value: the highest-ranked 10% contains 18.6% of validation attrition cases and has 1.86 times the population attrition concentration. The overlapping score distributions and 0.621 ROC-AUC show that the model is not certain about individual outcomes.